In [2]:
import numpy as np
import torch

In [ ]:
def ssim(mu_obs, mu_pred, sigma_obs, sigma_pred, sigma_cross, C1=0.01**2, C2=0.03**2):
    """
    Compute the Structural Similarity Index (SSIM) between two images.

    Parameters:
    - mu_obs: Mean of the observed image.
    - mu_pred: Mean of the predicted image.
    - sigma_obs: Variance of the observed image.
    - sigma_pred: Variance of the predicted image.
    - sigma_cross: Covariance between the observed and predicted images.
    - C1: Constant to stabilize the division with weak denominator (default: 0.01^2).
    - C2: Constant to stabilize the division with weak denominator (default: 0.03^2).

    Returns:
    - ssim_value: The computed SSIM value.
    """
    numerator = (2 * mu_obs * mu_pred + C1) * (2 * sigma_cross + C2)
    denominator = (mu_obs**2 + mu_pred**2 + C1) * (sigma_obs + sigma_pred + C2)
    
    ssim_value = numerator / denominator
    return ssim_value

def ssim_loss(y_true, y_pred):
    """
    Compute the SSIM loss between two images.

    Parameters:
    - y_true: The ground truth image.
    - y_pred: The predicted image.

    Returns:
    - loss: The computed SSIM loss.
    """
    mu_obs = np.mean(y_true)
    mu_pred = np.mean(y_pred)
    
    sigma_obs = np.var(y_true)
    sigma_pred = np.var(y_pred)
    
    sigma_cross = np.mean((y_true - mu_obs) * (y_pred - mu_pred))
    
    ssim_value = ssim(mu_obs, mu_pred, sigma_obs, sigma_pred, sigma_cross)
    
    # SSIM loss is defined as 1 - SSIM
    loss = 1 - ssim_value
    return loss

def get_weights(y_true):
    """
    Compute the weights for the loss function based on the ground truth image.

    Parameters:
    - y_true: The ground truth image.

    Returns:
    - w: The computed weights.
    """

    weights = torch.where(
        y_true < 5.0, 0.5,
        torch.where(
            y_true < 10.0, 1.0,
            torch.where(
                y_true < 20.0, 2.0,
                torch.where(
                    y_true < 30.0, 5.0,
                    torch.where(
                        y_true < 40.0, 10.0, 20.0
                    )
                )
            )
        )
    )
    return weights

def mse_weighted_loss(y_true, y_pred, w):
    """
    Compute the Mean Squared Error (MSE) loss between two images.

    Parameters:
    - y_true: The ground truth image.
    - y_pred: The predicted image.
    - w: Weighting factor for the MSE loss.

    Returns:
    - loss: The computed weighted MSE loss.
    """
    loss = w(y_true) * np.mean((y_true - y_pred) ** 2)
    return loss

def mae_weighted_loss(y_true, y_pred, w):
    """
    Compute the Mean Absolute Error (MAE) loss between two images.

    Parameters:
    - y_true: The ground truth image.
    - y_pred: The predicted image.
    - w: Weighting factor for the MAE loss.

    Returns:
    - loss: The computed MAE loss.
    """
    loss = w(y_true) * np.mean(np.abs(y_true - y_pred))
    return loss

def weighted_loss(y_true, y_pred, alpha=0.5,gamma=0.1):
    """
    Compute a weighted loss combining MSE and SSIM.

    Parameters:
    - y_true: The ground truth image.
    - y_pred: The predicted image.
    - alpha: Weighting factor for MSE (default: 0.5).
    - gamma: Weighting factor for SSIM (default: 0.1)

    Returns:
    - loss: The computed weighted loss.
    """
    mse = mse_weighted_loss(y_true, y_pred, get_weights(y_true))
    ssim_l = ssim_loss(y_true, y_pred)
    
    loss = alpha * mse + (1 - alpha) * gamma * ssim_l
    return loss

def dinamic_weighted_loss(y_true, y_pred, alpha=0.5, gamma=0.1):
    """
    Compute a dynamic weighted loss that adjusts the weighting based on the current loss values.

    Parameters:
    - y_true: The ground truth image.
    - y_pred: The predicted image.
    - alpha: Initial weighting factor for MSE (default: 0.5).
    - gamma: Initial weighting factor for SSIM (default: 0.1)
    Returns:
    - loss: The computed dynamic weighted loss.
    """
    mse = mse_weighted_loss(y_true, y_pred, get_weights(y_true))
    ssim_l = ssim_loss(y_true, y_pred)
    
    # Adjust alpha based on the current loss values
    if mse > ssim_l:
        alpha = min(alpha + 0.1, 1.0)  # Increase weight for MSE
    else:
        alpha = max(alpha - 0.1, 0.0)  # Decrease weight for MSE
    
    loss = 
    return loss

In [4]:
get_weights(torch.tensor([10.0]))

tensor([2.])